In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim 
import torchvision 
from torchvision.datasets import CIFAR10

In [2]:
# Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms 

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5,0.5,0.5))
    ]
)
trainset= CIFAR10(root='./data', train=True, download=True, transform=transform)
testset= CIFAR10(root='./data', train=False, download=True, transform=transform)

In [3]:
trainloader= DataLoader(trainset, batch_size=64, shuffle=True)
testloader=DataLoader(testset, batch_size=64, shuffle=False) 

# Cnn

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # kernel size=2, stride=2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers= nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x= self.conv_layers(x)
        x= x.view(x.size(0), -1) #flatterning
        x= self.fc_layers(x)
        return x


In [5]:
model= CNN()
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters())

# Training CNN

In [6]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs=model.forward(images)
        loss= criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()
    print(f"Epoch {epoch}, Training Loss: {epoch_training_loss/len(trainloader)}")

Epoch 0, Training Loss: 1.365828034899119
Epoch 1, Training Loss: 0.9373274178379942
Epoch 2, Training Loss: 0.7541098601525397
Epoch 3, Training Loss: 0.6331660397674727
Epoch 4, Training Loss: 0.5239890925490948
Epoch 5, Training Loss: 0.43086893183877095
Epoch 6, Training Loss: 0.3469937198111773
Epoch 7, Training Loss: 0.2750028537876923
Epoch 8, Training Loss: 0.21732157052916182
Epoch 9, Training Loss: 0.17182213815924763


In [8]:
# Evaluation
correct_labels=0
total_labels=0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs= model.forward(images)
        _, predicted= torch.max(outputs, 1)
        correct_labels +=(predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"Correct: {correct_labels}, Total: {total_labels}")
print(f"Accuracy: {correct_labels/total_labels*100:.2f}%")

Correct: 7506, Total: 10000
Accuracy: 75.06%
